# Configurations, presets and tuning

modde is configured with named **settings**: modules (e.g. `mutation_base`) and
numeric parameters (e.g. `lambda0`). This notebook shows how to set them,
inspect them, use named presets, and work with the configuration space for
automated algorithm configuration. All settings are listed in [Parameters](../parameters.md).

In [ ]:
import numpy as np
import ioh
import modde

def sphere(x):
    return float(np.sum(x ** 2))

## Settings by name

Module options are strings (case-insensitive).

In [ ]:
opt = modde.ModularDE(sphere, budget=3000, dim=5, lb=-5, ub=5,
                      mutation_base="target", mutation_reference="pbest",
                      use_archive=True, adaptation_method_F="SHADE", lambda0=40)
s = opt.settings
print("lambda0:", s.lambda0, s.provenance["lambda0"])
print("CR0:", s.CR0, s.provenance["CR0"])
print("archive_rate:", s.archive_rate, s.provenance["archive_rate"])

Values you set are never changed. Conflicts raise an error, and settings that
have no effect with the chosen modules produce a warning:

In [ ]:
try:
    modde.ModularDE(sphere, budget=100, dim=5, lb=-5, ub=5, mutation_n_comps=3, lambda0=6)
except ValueError as error:
    print("error:", error)

import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    modde.ModularDE(sphere, budget=100, dim=5, lb=-5, ub=5, memory_size=10)
print("warning:", caught[0].message)

## Presets

Popular DE variants are available by name. `exact=False` marks approximations
(components of the original paper that are not available as modules).

In [ ]:
print(modde.preset_names())
print(modde.preset_settings("L-SHADE", dim=10))

## The configuration space

`get_configspace(dim)` returns a [ConfigSpace](https://automl.github.io/ConfigSpace/)
`ConfigurationSpace` with every module and tunable parameter. Module-specific
parameters are *conditional*: they only appear when their module is active.
Its default configuration is classic DE, and every configuration in it is valid.

In [ ]:
cs = modde.get_configspace(dim=5)
print(len(list(cs.values())), "hyperparameters,", len(cs.conditions), "conditions")
print(cs.get_default_configuration())

Presets are configurations of this space, so they can be edited and used as
starting points for a tuner:

In [ ]:
config = modde.get_presets(dim=5)["L-SHADE"]
values = dict(config)
values["archive_rate"] = 1.5
config = type(config)(cs, values=values)
res = modde.fmin(sphere, budget=3000, dim=5, lb=-5, ub=5, config=config, seed=1)
print("edited L-SHADE:", res.f)

## A tiny tuning loop

Any tuner that speaks ConfigSpace (e.g. [SMAC](https://github.com/automl/SMAC3))
can tune modde directly. Here, plain random search on a BBOB function shows the
workflow; `settings_from_config` turns a configuration into settings.

In [ ]:
def evaluate(config, seed=0):
    problem = ioh.get_problem(15, instance=1, dimension=5)
    opt = modde.ModularDE(problem, budget=2000, config=config, seed=seed).run()
    return opt.best_f - problem.optimum.y

cs.seed(1)
candidates = [cs.get_default_configuration()] + list(cs.sample_configuration(8))
scores = [np.mean([evaluate(c, seed) for seed in range(3)]) for c in candidates]
best = candidates[int(np.argmin(scores))]
print("default (classic DE):", round(scores[0], 3))
print("best of the sample:  ", round(min(scores), 3))
print({k: v for k, v in dict(best).items() if k in ("mutation_base", "crossover", "adaptation_method_F", "lambda0")})

## Tuning with irace

For [irace](https://mlopez-ibanez.github.io/irace/), `modde.irace` writes the
parameter file (with all conditions) and an initial-configurations file with
the default and all presets. In the target runner, `config_from_irace` turns
irace's command-line arguments back into a configuration.

In [ ]:
import tempfile
paths = modde.irace.write_irace_files(tempfile.mkdtemp(), dim=5)
print("".join(paths["parameters"].read_text().splitlines(keepends=True)[:8]))

config = modde.irace.config_from_irace("--mutation_base TARGET --mutation_reference PBEST --pbest_sampling FIXED --pbest_rate 0.1", cs)
print(config["mutation_base"], config["pbest_rate"])